In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os, ast, sqlite3, re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

In [2]:
def cleanText(text):
    return re.sub(r'[^A-Za-z0-9]', '', text).strip().lower()

In [3]:
df = pd.read_csv('DataBase.csv')
df['installation_date'] = pd.to_datetime(df.installation_date, errors='coerce')
df['timestamp'] = pd.to_datetime(df.timestamp, errors='coerce')
df

,id,sensor_id,series_num,suntech,timestamp,updated_at,installed,telemetry,company,plate,installation_date
0,MIC8767790427121753,ETL9263776954081852,4,1.700004e+09,2026-01-06,2026-04-28 19:16:17.634201,True,suntech,sighir,MALETA_SIGHIR,2026-01-21
1,MIC7952031891471590,ETL7952031891471590,6,NaN,2026-03-18,2026-03-18 11:51:29.539832,True,mix,predileto,RJO6I53,2026-03-18
2,MIC1009016058992021,ETL8355798954801671,12,NaN,2024-07-05,2025-12-10 19:36:52.528153,True,mix,predileto,LUA5E58,2024-07-12
3,MIC3982343538817967,ETL3178405820036359,22,NaN,2024-09-02,2025-12-10 19:36:52.528153,True,mix,manchur,KYA9508,2024-09-23
4,MIC1700087363643403,ETL0644031280801291,25,NaN,2024-08-14,2025-12-10 19:36:52.528153,True,mix,logika,SRJ7J95,2024-07-15
...,...,...,...,...,...,...,...,...,...,...,...
117,MIC6628888901721325,ETL5501447410121100,303,NaN,2026-04-29,2026-04-29 18:32:18.695447,True,mix,predileto,SSC2A28,2026-05-12
118,MIC5219749327191044,ETL558097971095111,305,NaN,2026-04-30,2026-04-30 13:11:12.459117,True,mix,predileto,RKG9C05,2026-05-12
119,MIC5949938607321190,ETL316616922406633,307,NaN,2026-04-30,2026-04-30 14:03:55.013456,True,mix,predileto,SQV6D04,2026-05-05
120,MIC4169182339378341,ETL110290530804220,308,NaN,2026-04-30,2026-04-30 17:45:00.774715,True,mix,predileto,RJT9G28,2026-05-12


In [4]:
df = df.loc[(df.company != 'logika')]
df

,id,sensor_id,series_num,suntech,timestamp,updated_at,installed,telemetry,company,plate,installation_date
0,MIC8767790427121753,ETL9263776954081852,4,1.700004e+09,2026-01-06,2026-04-28 19:16:17.634201,True,suntech,sighir,MALETA_SIGHIR,2026-01-21
1,MIC7952031891471590,ETL7952031891471590,6,NaN,2026-03-18,2026-03-18 11:51:29.539832,True,mix,predileto,RJO6I53,2026-03-18
2,MIC1009016058992021,ETL8355798954801671,12,NaN,2024-07-05,2025-12-10 19:36:52.528153,True,mix,predileto,LUA5E58,2024-07-12
3,MIC3982343538817967,ETL3178405820036359,22,NaN,2024-09-02,2025-12-10 19:36:52.528153,True,mix,manchur,KYA9508,2024-09-23
7,MIC3004645420106011,ETL2608402025435219,32,1.700007e+09,2024-09-06,2026-02-05 13:52:53.033824,True,suntech,predileto,RJT5E02,2026-02-06
...,...,...,...,...,...,...,...,...,...,...,...
117,MIC6628888901721325,ETL5501447410121100,303,NaN,2026-04-29,2026-04-29 18:32:18.695447,True,mix,predileto,SSC2A28,2026-05-12
118,MIC5219749327191044,ETL558097971095111,305,NaN,2026-04-30,2026-04-30 13:11:12.459117,True,mix,predileto,RKG9C05,2026-05-12
119,MIC5949938607321190,ETL316616922406633,307,NaN,2026-04-30,2026-04-30 14:03:55.013456,True,mix,predileto,SQV6D04,2026-05-05
120,MIC4169182339378341,ETL110290530804220,308,NaN,2026-04-30,2026-04-30 17:45:00.774715,True,mix,predileto,RJT9G28,2026-05-12


# TELEMETRIA MIX - COMPARAÇÃO

In [5]:
mix_df = pd.read_csv('files/Detailed Event Report.csv')
mix_df.head()

/tmp/ipykernel_44885/2085577859.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  mix_df = pd.read_csv('files/Detailed Event Report.csv')


,AssetSiteName,ReportGroup,AssetName,AssetExtra,AssetID,Asset2SiteName,Asset2ReportGroup,AssetName2,AssetExtra2,AssetID2,FleetNumber,TripID,EventKey,EventDescription,EventType,EventStartDate,EventStartTime,EventEndDate,EventEndTime,TimeZone,TotalOccurs,EventValue,MeasurementUnits,RoadSpeedLimit,EventLocation,TotalDuration,StartOdo,EndOdo,F_StartStreet,F_StartSuburb,F_StartCity,F_StartTown,F_StartRegion,F_EndStreet,F_EndSuburb,F_EndCity,F_EndTown,F_EndRegion,StartLocation,EndLocation,StartLatLong,EndLatLong,FuelUsed,DistanceCalculatedFromOdo
0,MACAÉ,NaN,RKH8A44,RKH8A44,74,Default Site,NaN,Unknown,NaN,0,CAVALO 6X2,#Error,3433360518641462441,Etilometro Sighir - Not active communication,Notify,01/05/2025,02:12:19,01/05/2025,02:12:19,BRT,1,"0,00",NaN,NaN,NaN,00:00:01,"0,00","0,00",Avenida Lacerda Agostinho,Barra de Macaé,Macaé,[UNKNOWN],Rio de Janeiro,[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,BASE 2 MACAÉ,NaN,"-22.33699 , -41.77841",NaN,"0,000","0,0"
1,MACAÉ,NaN,RKH8A44,RKH8A44,74,Default Site,NaN,Unknown,NaN,0,CAVALO 6X2,#Error,3433425814609798048,Etilometro Sighir - Not active communication,Notify,01/05/2025,06:25:42,01/05/2025,06:25:42,BRT,1,"0,00",NaN,NaN,NaN,00:00:01,"0,00","0,00",Avenida Lacerda Agostinho,Barra de Macaé,Macaé,[UNKNOWN],Rio de Janeiro,[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,BASE 2 MACAÉ,NaN,"-22.33699 , -41.77841",NaN,"0,000","0,0"
2,MACAÉ,NaN,RKH8A44,RKH8A44,74,Default Site,NaN,Unknown,NaN,0,CAVALO 6X2,#Error,3433426874251912498,Etilometro Sighir - block status 0,Notify,01/05/2025,06:29:49,01/05/2025,06:29:49,BRT,1,"0,00",NaN,NaN,NaN,00:00:01,"0,00","0,00",Avenida Lacerda Agostinho,Barra de Macaé,Macaé,[UNKNOWN],Rio de Janeiro,[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,BASE 2 MACAÉ,NaN,"-22.33699 , -41.77841",NaN,"0,000","0,0"
3,MACAÉ,NaN,RKH8A44,RKH8A44,74,MACAÉ,NaN,LUCIAN PEREIRA LIMA - DESLIGADO,NaN,158,CAVALO 6X2,#Error,3433427301949984358,Etilometro Sighir - block status 0,Notify,01/05/2025,06:31:28,01/05/2025,06:31:28,BRT,1,"0,00",NaN,NaN,NaN,00:00:01,"0,00","0,00",[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,NaN,NaN,NaN,NaN,"0,000","0,0"
4,MACAÉ,NaN,RKH8A44,RKH8A44,74,MACAÉ,NaN,LUCIAN PEREIRA LIMA - DESLIGADO,NaN,158,CAVALO 6X2,#Error,3433427302857117884,Etilometro Sighir - Not active communication,Notify,01/05/2025,06:31:28,01/05/2025,06:31:28,BRT,1,"0,00",NaN,NaN,NaN,00:00:01,"0,00","0,00",[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,[UNKNOWN],[UNKNOWN],[UNKNOWN],[UNKNOWN],NaN,NaN,NaN,NaN,NaN,"0,000","0,0"


In [6]:
mix = mix_df.AssetName.unique()
print(f'{len(mix)} no servidor da MIX')
mix

51 no servidor da MIX


array(['RKH8A44', 'SRM9D84', 'PPQ9017', 'LUE9A29', 'RJO6I53', 'RKU6C18',
       'LUA5E58', 'SRV2A57', 'RJY7B05', 'SRP8B13', 'RJS5A13', 'RJZ4C27',
       'SRA4H41', 'RJZ6G46', 'RJM4H64', 'Teste Letícia', 'RKR8A05',
       'RKC9H51', 'RJG6H37', 'RKL5C01', 'SQV7F79', 'QRM8G34', 'SRV2A94',
       'RJX3J88', 'RJX6I60', 'RJO7B87', 'KZI3171', 'QRM8G13', 'TTM3D82',
       'SRM9F81', 'RJN8B51', 'SRN1B14', 'SRO7I44', 'RKD4I49', 'RJE6F03',
       'RKO8A59', 'SRD5I52', 'RJP4I71', 'RJT5E02', 'SSD0C26', 'RJR7C09',
       'Teste etilometro 1', 'RKG5D43', 'RKV6J92', 'RKC6G52', 'SRS7J58',
       'QRM7G63', 'QRH8I97', 'RJE2J65', 'EVW4H85', 'SSA7F80'],
      dtype=object)

In [7]:
sighir = df.plate.unique()
print(f'{len(sighir)} no servidor Sighir')
sighir

117 no servidor Sighir


array(['MALETA_SIGHIR', 'RJO6I53', 'LUA5E58', 'KYA9508', 'RJT5E02',
       'PPQ9017', 'SRV2A57', 'CP027', 'MOV1T23', 'RKU6C18', 'PC483',
       'CS004', 'CR009', 'CS002', 'CS001', 'SRO0A73', 'RE 624', 'CS008',
       'CS007', 'CS006', 'CR52', 'CS003', 'Mosaic', 'C948', 'C986',
       'MN870', 'C1112', 'CS005', 'CR27', 'C 958', 'CP004', 'CP001',
       'RE624', 'MN869', 'CS011', 'CR 34', 'CR018', 'C 948', 'SQV3I75',
       'BANCADA', 'RKG9C70', 'RJY4B66', 'SRK8F90', 'RJX3J88', 'RKH8A44',
       'RJZ4C27', 'RJX6I60', 'RKO8A59', 'SQV7F79', 'SRN1B14', 'RKR8A05',
       'RKL5C01', 'RJS5A13', 'RJG6H37', 'RJZ6G46', 'RKC9H51', 'EMPRIO',
       'EOJ5I60', 'QRH8I97', 'BEI5H39', 'RHI3C91', 'FGT9B86', 'SPX4A02',
       'SEQ3J35', 'JAO1G85', 'SRA4H41', 'SRV2A94', 'SRE5D58', 'QRM8G34',
       'SQZ7I20', 'EMP009', 'SRU4A50', 'TTD7F73', 'RJA7A37', 'KZI3171',
       'SRQ8A62', 'SRP8B13', 'RJM4H64', 'SSC2A45', 'RJV1A55', 'SRM9F81',
       'JBL6F29', 'TUL2A69', 'JBC1E74', 'SQX8C74', 'CS010', 'RJE2J65',
 

In [8]:
diference = [plate for plate in mix if plate not in sighir and 'teste' not in plate.lower()]
print(f'{len(diference)} etilometros não instalados no servidor sighir')
diference

7 etilometros não instalados no servidor sighir


['SRM9D84', 'RJY7B05', 'RJO7B87', 'RJN8B51', 'SRO7I44', 'RKD4I49', 'EVW4H85']

# ANÁLISE MIX

In [9]:
mix_df.EventDescription.unique()

array(['Etilometro Sighir - Not active communication',
       'Etilometro Sighir - block status 0', 'Etilometro Sighir - Log',
       'Etilometro Sighir - Active communication',
       'Etilometer Sighir - Sensor inicializado (boot)',
       'Etilometer Sighir - Sopro identificado',
       'Etilometer Sighir - Leitura realizada',
       'Etilometro Sighir - Liberado',
       'Etilometer Sighir - Liberado (log)',
       'Etilometer Sighir - Dispositivo inicializado',
       'Etilômetro Sighir - álcool não detectado',
       'Etilometer Sighir - Aviso de teste randômico',
       'Etilometer Sighir - Leitura adiada',
       'Etilometer Sighir - Sem sopro',
       'Etilometer Sighir - Bloqueado (log)',
       'Etilometro SIghir - Bloqueado',
       'Etilômetro Sighir - álcool detectado',
       'Etilômetro Sighir - temperatura alta',
       'Etilometro - tentativa de ativação',
       'Etilometro - ativação bem sucedida', 'Etilometro - não ativado',
       'Etilometer Sighir - concentração